# TESTING.md — how to validate `order_execution_v2.ipynb` gradually

The notebook ships configured for the **synthetic `COVID` asset** so it runs
end-to-end anywhere, with no real data. Escalate in the order below; each tier
changes only **Section 0**, then *Run All*.

---

## 0. One-time: generate the synthetic data

```bash
python make_test_data.py .
```

This writes into `data/Test/`:

| File | Used by asset | Purpose |
|------|---------------|---------|
| `TINYH20.csv`, `AIAgent_Tiny.csv`     | `TINY`  | ~10 days, clean — 30-second smoke test |
| `SMALLH20.csv`, `SMALLM20.csv`, `AIAgent_Small.csv` | `SMALL` | 2 contracts + **missing minutes** — exercises rollover (Sec 2) and the time-based windowing fix (Sec 3, FIX 4) |
| `COVIDH20.csv`, `AIAgent_Covid.csv`   | `COVID` | ~4 months with a volatility-regime spike — exercises state binning and gives non-trivial results (shipped default) |

Your real folders sit next to it:

```
data/
├── Nasdaq/  NQH20.csv NQM20.csv NQU20.csv  AIAgent_Nasdaq.csv
├── Gold/    GCG24.csv GCJ24.csv GCM24.csv GCQ24.csv  AIAgent_Gold.csv
└── Test/    (synthetic, above)
```

---

## Tier 1 — TINY (fast smoke test, ~30 s)

**Section 0:**
```python
ASSET           = 'TINY'
BURN_IN         = 100      # TINY is short; 500 would leave no training data
TEST_START_FRAC = 0.5
```
Run All. You're checking that *every cell executes*, not the numbers. Expect a
handful of trades. Confirm:
- Section 3 prints `[FIX 4 check] ... off the 5-min grid: 0`
- Section 6 prints `[FIX 2 check] ... MATCH — no leakage`

> If you see `in-sample valid bars = 0`, `BURN_IN` is ≥ the in-sample window —
> lower it.

---

## Tier 2 — SMALL (rollover + data gaps, ~1 min)

**Section 0:**
```python
ASSET           = 'SMALL'
BURN_IN         = 150
TEST_START_FRAC = 0.6
```
`SMALL` has two contracts and 3% randomly-dropped minutes. This is the tier that
proves the two structural fixes:
- **Rollover (Sec 2):** the load log should show both `SMALLH20` and `SMALLM20`,
  stitched at the volume crossover (~mid-Feb).
- **FIX 4 (Sec 3):** despite the gaps, `[FIX 4 check]` must still report `0` bars
  off the τ-grid. (With the old row-based windowing this is where bars would
  silently widen.)

---

## Tier 3 — COVID (full pipeline, regime diversity, ~3–5 min)

**Section 0 (this is the shipped default):**
```python
ASSET           = 'COVID'
BURN_IN         = 300
TEST_START_FRAC = 0.60
```
Now the numbers matter. Expect ~150–200 trades and a fill rate near the 0.60
threshold. Read the Section 10a summary and confirm `net_vs_market` and
`realized_avg_all` are **different** — on synthetic COVID the gap is large
(violent intra-bar swings), which is the whole point of FIX 5.

---

## Tier 4 — real NQ (the primary deliverable)

**Section 0:**
```python
ASSET           = 'NQ'
BURN_IN         = 500      # README's conservative value for the 5-month window
TEST_START_FRAC = 0.60
VOL_PROXY       = 'ewmv'   # try 'range_mean' to compare (FIX 1)
MC_OBJECTIVE    = 'expected_ticks'   # try 'cond_on_fill' to compare (FIX 3)
```
`TICK_SIZE` is resolved automatically from the registry — you do **not** hand-set
it. Run All. Compare your headline numbers against the README's
(fill rate ≈ 66%, net_vs_market ≈ +2.2 ticks). They will differ from v1 because
the ePDF is no longer contaminated by test data (FIX 2) — expect v2 to be a bit
**less** optimistic. That difference is the correction.

---

## Tier 5 — real GC (cross-asset generalization)

**Section 0:** `ASSET = 'GC'` (tick auto-resolves to 0.10). Run All. The Section
10e cross-asset cell will compare `['NQ', 'GC']` automatically once `ASSET` is a
real asset.

---

## Performance notes

- **Slowest cell = Section 10d sensitivity.** It re-runs the whole pipeline for
  each `(τ, threshold)` pair. The grids are trimmed by default
  (`TAU_GRID=[5,10,15]`, `THRESH_GRID=[0.5,0.6,0.7]`); widen them for the final
  report. Sensitivity/cross-asset runs skip the bootstrap (`bootstrap_reps=0`)
  because they only need `ℓ*`, not the CI band.
- If a real run is slow while iterating, set `BOOTSTRAP_REPS = 200` temporarily
  (Section 0) — it only affects the width of the Section 7 CI band.

---

## What to spot-check when reviewing each fix

| Fix | Where to look | What "correct" looks like |
|-----|---------------|---------------------------|
| FIX 2 | Section 6 print | `in-sample valid bars == store total`, and store total < the bar count you'd get training over the whole set |
| FIX 4 | Section 3 print | `off the τ-min grid: 0`, even on `SMALL`/real data with gaps |
| FIX 5 | Section 10a | `net_vs_market` (alpha) and `realized_avg_all` are two separate, differently-valued lines |
| FIX 1 | Section 5 | flip `VOL_PROXY`; volatility-state distribution shifts |
| FIX 3 | Section 7 | flip `MC_OBJECTIVE`; `ℓ*` table may change (often identical — see the §7 note) |
| FIX 6 | Section 4/5 | `delta_x` is `open.diff()`; `dir_state` uses its lag-1 value |
